# Coevolution - Graham version gLM2
Adapted from @sokrypton

In [ ]:
TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'
import os
from typing import List
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import bokeh.plotting
from bokeh.transform import linear_cmap
from bokeh.plotting import figure, show
from bokeh.palettes import viridis
from transformers import AutoTokenizer, AutoModelForMaskedLM
from matplotlib.colors import to_hex

bokeh.io.output_notebook()

In [ ]:
#@markdown ## Flags
fast = True # @param {type:"boolean"}
#@markdown - only perturb the `mask` token

In [ ]:
#@markdown ## setup gLM2_650M
import os
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_NAME = "tattabio/gLM2_650M"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
NUC_TOKENS = tuple(range(29, 33)) # 4 nucleotides a,t,c,g
AA_TOKENS = tuple(range(4,24)) # 20 amino acids
NUM_TOKENS = len(AA_TOKENS) + len(NUC_TOKENS)

model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME, trust_remote_code=True).eval().to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
MASK_TOKEN_ID = tokenizer.mask_token_id

from bokeh.models import Span

def create_figure(contact_df, tokens: List[str], split_idx=None):
    tools = "hover,save,pan,box_zoom,reset,wheel_zoom"
    seqlen = len(tokens)

    contact_df['i_token'] = contact_df['i'] + ': ' + contact_df['i'].astype(int).map(lambda x: tokens[x-1])
    contact_df['j_token'] = contact_df['j'] + ': ' + contact_df['j'].astype(int).map(lambda x: tokens[x-1])

    p = figure(title="COEVOLUTION",
               x_range=[str(x) for x in range(1,seqlen+1)],
               y_range=[str(x) for x in range(1,seqlen+1)][::-1],
               width=800, height=800,
               tools=tools, toolbar_location='below',
               tooltips=[('i', '@i_token'), ('j', '@j_token'), ('value', '@value')])

    cmap = plt.colormaps["Blues"]
    blues = [to_hex(cmap(i)) for i in np.linspace(0, 1, 256)]

    p.rect(x="i", y="j", width=1, height=1, source=contact_df,
           fill_color=linear_cmap('value', blues,
                                  low=contact_df.value.min(),
                                  high=np.percentile(contact_df.value, 99)),
           line_color=None)

    if split_idx is not None:
        split_str = str(split_idx)

        vline = Span(location=split_str,
                     dimension='height',
                     line_color='red',
                     line_width=3)

        hline = Span(location=split_str,
                     dimension='width',
                     line_color='red',
                     line_width=3)

        p.add_layout(vline)
        p.add_layout(hline)

    p.xaxis.visible = False
    p.yaxis.visible = False

    return p
def contact_to_dataframe(con):
  sequence_length = con.shape[0]
  idx = [str(i) for i in np.arange(1, sequence_length + 1)]
  df = pd.DataFrame(con, index=idx, columns=idx)
  df = df.stack().reset_index()
  df.columns = ['i', 'j', 'value']
  return df

def jac_to_contact(jac, symm=True, center=True, diag="remove", apc=True):

  X = jac.copy()
  Lx,Ax,Ly,Ay = X.shape

  if center:
    for i in range(4):
      if X.shape[i] > 1:
        X -= X.mean(i,keepdims=True)

  contacts = np.sqrt(np.square(X).sum((1,3)))

  if symm and (Ax != 20 or Ay != 20):
    contacts = (contacts + contacts.T)/2

  if diag == "remove":
    np.fill_diagonal(contacts,0)

  if diag == "normalize":
    contacts_diag = np.diag(contacts)
    contacts = contacts / np.sqrt(contacts_diag[:,None] * contacts_diag[None,:])

  if apc:
    ap = contacts.sum(0,keepdims=True) * contacts.sum(1, keepdims=True) / contacts.sum()
    contacts = contacts - ap

  if diag == "remove":
    np.fill_diagonal(contacts,0)

  return contacts


def get_categorical_jacobian(sequence: str, fast: bool = False):
  all_tokens = NUC_TOKENS + AA_TOKENS
  num_tokens = len(all_tokens)

  input_ids = torch.tensor(tokenizer.encode(sequence), dtype=torch.int)
  tokens = tokenizer.convert_ids_to_tokens(input_ids)
  seqlen = input_ids.shape[0]
  # [seqlen, 1, seqlen, 1].
  is_nuc_pos = torch.isin(input_ids, torch.tensor(NUC_TOKENS)).view(-1, 1, 1, 1).repeat(1, 1, seqlen, 1)
  # [1, num_tokens, 1, num_tokens].
  is_nuc_token = torch.isin(torch.tensor(all_tokens), torch.tensor(NUC_TOKENS)).view(1, -1, 1, 1).repeat(1, 1, 1, num_tokens)
  # [seqlen, 1, seqlen, 1].
  is_aa_pos = torch.isin(input_ids, torch.tensor(AA_TOKENS)).view(-1, 1, 1, 1).repeat(1, 1, seqlen, 1)
  # [1, num_tokens, 1, num_tokens].
  is_aa_token = torch.isin(torch.tensor(all_tokens), torch.tensor(AA_TOKENS)).view(1, -1, 1, 1).repeat(1, 1, 1, num_tokens)

  input_ids = input_ids.unsqueeze(0).to(DEVICE)
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=True):
    f = lambda x:model(x)[0][..., all_tokens].cpu().float()

    x = torch.clone(input_ids).to(DEVICE)
    ln = x.shape[1]

    fx = f(x)[0]
    if fast:
      fx_h = torch.zeros((ln, 1 , ln, num_tokens), dtype=torch.float32)
    else:
      fx_h = torch.zeros((ln,num_tokens,ln,num_tokens),dtype=torch.float32)
      x = torch.tile(x,[num_tokens,1])
    with tqdm(total=ln, bar_format=TQDM_BAR_FORMAT) as pbar:
      for n in range(ln): # for each position
        x_h = torch.clone(x)
        if fast:
          x_h[:, n] = MASK_TOKEN_ID
        else:
          x_h[:, n] = torch.tensor(all_tokens)
        fx_h[n] = f(x_h)
        pbar.update(1)
    jac = fx_h-fx
    valid_nuc = is_nuc_pos & is_nuc_token
    valid_aa = is_aa_pos & is_aa_token
    # Zero out other modality
    jac = torch.where(valid_nuc | valid_aa, jac, 0.0)
    contact = jac_to_contact(jac.numpy())
  return jac, contact, tokens

## Protein-Protein Interaction

In [ ]:
A = "MDNINGALAFKATLDINDFNVSAQAMERSIKQVSSTAVSESSVMDNSIQSFAQNGAKYIV SYLVGQGMGTLLQSIVQTRGQFQQLEIAFTTMLKSGTQAKGLMDRLIDTAAKTPFDLSGI ASSAKQMLAYGSTVDNVVDELVMLGNVASGVGAPLQDIAYLYGTLRTQGRAFTVDIRQFA GRGIPIYEELAKVLGVTKDEVSNLVTEGKVGFAEVEKAFQNMTGKAGTYYNLMQEQSKSL TGMISNMGDAWEQSLNKLGADNQDVFAGAIESATYMAEHLDDILRILKAVAIGYGSVKAA IVLNTLATKGYTGVALLDNTARQAKISLMKLEAVATGQMVAQTKAMVAAQNSHVAALQAQ LTVEEHANMVKQLRIATIQQMLTIQQAEYLSNLNLTASSANYEAVALSVLTVEQKQALSK LDLSAKSAVYRAALENEVAVKTQNSAATLNAMRTDVKAAAVKMESARADALAAKAAVERA YLEVYRAQQTGNAEKIAIATKKMEAAEDNAAIARKTALATQSDFYAKKKLLEATATKQST AASVADTTAKTTQGAVTSVLTAITTKATVAVKALWASMMSNPIGWVMGLIGALVSVITLF TGKQKEATTATGEFQDTTKKEIDDLNLLFAVLQNTEKGTQTHKNTIEKINAVCKQYNKTL LDENATLDLQRLKYEELTTAIQQTTAEKIKAKYTEQAMQELVQSQTDALDKLKENAEDAT YKEIQEVMETTPEGVTVMMNKVVDVASSSIRGASGAVWDAVESMAVESANQLKGLTGQAY TDAFNNSLNSIVSQVKQSTSATDKEMDAFKSNIKTYLESIVQSAKKADETIGKVNKELEA FIAPKDTTSVTESTDYVAMSFNELDKKIGETQKSIDTLNAKKVKVEADNTQLKELKDLLD KLNGAVNTKTTNLNTEKGISDRIKQLKDLREAAIIGSSDYKSYDTQIKKLEARLPKHTTG DKADSAAKQLRERQLEADRKLEADRIAVLEEGYEKRKRTLSLQHKEALDNIDKEEKALAK ARKDAGKGGLSKSEKDGFDERRTLENKKYDKAQNKLFDGEIEYKKQQYALYFRWVRNMGE DVANTQFATLLKGGKSYKEYVENEIKALKDKQQAGTLTEGESNQLISLNMQYNEITGAKS AMDSFKESVSKTIQQAQTLAEKLEAIADAKEKLANGSSGLVGADEKAEATLFVSEEDEKL QEEVQQKILSSYRTFEEQRNDIQKEYALLRAAAQKTGDQERINQVNKSEAEALSTLTANM LKQSDSWKKLFGDLDSLSVAEIDKLVADIETKLKDADLKLNPVDYRALIDSLNQAKETLI SKNPFKALGTFYDDYIEAKKKLAEAKANVAAGKGTDEDVKKAEADMKKAAKGVTKSIETI TDTATTCGNAIASMFSDLGQDDLANGLGTAMELFGQLGNAAASVGKMMSGDILGGVTGMV SAVTSVVGIFAKLHDSKYEKKIQNLQKEIDALEQSYNRLERAYNNTYWVFNDSQREAYEK NIQLINDQIRALEQEANIAKKNWDFARYAQLNKEIKELNKQLKNAEENGDMFSIYEAQKK NLKQQQEDLRKQIQAEKDKKKTDNGKIQQWNEQIESITQQIEDLDRSMMETLAGTDVKTA IDEFADALVDAYCKGEDAAEALGEKTKEVLKKAVVEALKREFLAKGINDAVLYLGESMKD GKLTDVEKREFERMVNAAGDLFNSALEGIGDWIKDVEEETVQQDPLTGAVTSMSEETGGV IAGRLNAFVINQSDQTSIMRQALVYQAEIAANTKLSASELTEIKTTLKRIENKDSSLLSQ GIA" #@param
B = "MGTLEIIRRNGEKVRLFSKEPFCTLKSAAQNSSLMGDDNVQLSIVSSELLNLGKGDKIIVEGEEYTIRTKVNREMLSDNHYVHDATFYGVMYELMKSLYRNTDANGKSSKSTFDLTYNIRDFVKVLIYNVSRDYPGLWAFDEANCPDTEPRTISFARNNCLQVLQMLCSDREFDLEFLITQKDGVRTIHIGKFGAKVVPPGGNAFFEWGKGNGLYKLKEQKVDDKTIITRLWVEGGTTNIRSDYRDYSERLQLPFPVRLNKKEHRLWDGTIVPPQSEYIGISDDNKRYLEDGDLRDALGSDEDAVTYDNIFPKRTGTVTALVADDINSFIDDTMDFDLNEKDDKGTKYLINEVSAKITFISGKLAGQQFELAQKGGYDHATKRFTLIPFTDNRGLTIPTTESEAYRITEGDTYKITDIHLPKSYEDDAEEDLWYAGYNEFKPRTQARAQYQLTFERSYFLNALPSDSETTVFHVGNYVPVKDERFGIEKNIRIQKVTKNLLVEHDYTLTLSDITAISPITQTVVDVGRHETIIENNRLRDLTKARRGWRTTEELRTMVYDTDGYFDPENIKPNSIDTNMLTVGSKSQQFVLIDVILQANVNGISNRFDASAGVLAHLTIDDEIIKHWNMAAGSFTLSSPKGYYVFAKCSKKSTDGIWYVTQEQLKVEPTEDPNNYYFQVGILGSVHSDDDFRDFTTTYGFTRINGNTITTGKIITSDKECYLDLDGNKFRIGDSSSSIDWNVTAKRQLTLHNVRLLSDSGDTSHIGVFRGTYNPKYVYYAGDEVTYTVNGETCTYRYTNPTPSMGNLPTNSVYWSVVAKGSTGDKGESGLSVFYTYNDSETKPATPTGDGSTGGWHRTSSENVVWMSIKNAKTDTEGAWGIPFRVRGADGTSINIKGSKDNVSQLPTVGNSEGDAYLIGGNLYIWDGTNWKDVGAIKGEDGKSSYLHKKYSDDGGKTFTAGNGETPGRWLGLYVDMIPTDSDKPSAYKWSDTKGQDGTPGLPGEDGRTPYFHIKYSDNGGMSFTANNGEEPGDYIGQYTDYVQKDSDNPMDYTWALIKGESGTGGTDAGAGEYYEYRYAKNGSTSTPPTLDINSSNPAGWTTEMPTLDEFEYLWCIMAKKSGLADRTKFYLPIEENNFSDVSGNGYNGVLGSGTVVKDGTRYALNLSGGMESRIPYDLPFGESFTLCFWMKSEQRQIKWMLNGYNGRHYVEKSIAITPNTWFHLAFRFNDCTVTVFKNGEQLHSGSVNIMAVGFAIYDDDVFGSAVYFDDVRLLMGALPVNDIASVMNGKADTMVQKWSTPVRINSYNGDKGEPGTSVTLADVEYAQSTSNTVAPTTGWQTTAPTWIDGRYIWSRTKVSYSDNTTTYTKAVCITGGKGTTGVGVSSIVEQYYLSSSATSLLGGSWGTTRPTWKDKWYIWTRSVITYTNGTSDTTAAICVTGSKGDKGEPGDKGEKGDSPAAVYQGLYSSSKTYYGNKYRLDVVKYNNVFYVARIDAGTFSGVVPTNTSKWNTFGAQFESVATNLLLAENANIAGWVFRNGRLESQNGNVYMDGNKGEVRLKGTLQLSTGFSGNFSDVNLFYLPETTSERNITMGYEKEDIGKVCRLYNSGAFGKANYLVEMNTFSITDNVTSSSSSLYALIRPQEIVEMTCFERKGSTPSKILGEWTITSRFSRDDFITNGATGRFARIMAMGTFYGSSGASVSGKFFDGRNLREVFTASRQDTGRYRLSFANGTLPSGYMVMVCGNGTISGGSNPLKANVYVQYSTYFEVMTSDDSSRNDGNFNFVVFAPDWDYKY" #@param

sequence = f"<+>{A}<+>{B}"

J, contact, tokens = get_categorical_jacobian(sequence, fast=fast)

df = contact_to_dataframe(contact)

# ❗ Do NOT pass split_idx into the function
p = create_figure(df, tokens)

# ---- Add split lines BEFORE show() ----
split_idx = 3 + len(A)
seqlen = len(tokens)
split_str = str(split_idx)

p.line(
    x=[split_str, split_str],
    y=[str(1), str(seqlen)],
    line_color="red",
    line_width=3
)

p.line(
    x=[str(1), str(seqlen)],
    y=[split_str, split_str],
    line_color="red",
    line_width=3
)

# ---- Show only once ----
show(p)

In [ ]:
from bokeh.plotting import figure, show
from bokeh.transform import linear_cmap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex

def plot_A_vs_B(contact, tokens, len_A, major_step=10, minor_step=1):

    # ---- Identify clean biological regions (exclude "<+>") ----
    start_A = 3
    end_A = 3 + len_A
    start_B = end_A + 3
    end_B = len(tokens)

    contact_AB = contact[start_A:end_A, start_B:end_B]

    # ---- Convert to dataframe ----
    idx_A = [str(i) for i in range(1, contact_AB.shape[0] + 1)]
    idx_B = [str(i) for i in range(1, contact_AB.shape[1] + 1)]
    df_AB = pd.DataFrame(contact_AB, index=idx_A, columns=idx_B).stack().reset_index()
    df_AB.columns = ['i', 'j', 'value']

    idx_B_rev = idx_B[::-1]

    # ---- Create figure ----
    p = figure(
        title="A vs B Interactions",
        x_range=idx_A,
        y_range=idx_B_rev,
        width=800,
        height=800,
        tools="hover,save,pan,box_zoom,reset,wheel_zoom",
        tooltips=[("A position", "@i"), ("B position", "@j"), ("value", "@value")]
    )

    # ---- Major labels every `major_step`, minor ticks every `minor_step` ----
    p.xaxis.major_label_overrides = {
        val: val if (int(val)-1) % major_step == 0 else ""
        for val in idx_A
    }
    p.yaxis.major_label_overrides = {
        val: val if (int(val)-1) % major_step == 0 else ""
        for val in idx_B_rev
    }

    # Optionally adjust minor ticks to appear as smaller marks (no labels)
    # For categorical axes, Bokeh draws a tick per factor automatically
    # So major labels + minor ticks are naturally drawn by leaving empty strings on minor ticks

    p.xaxis.minor_tick_line_color = "gray"  # Optional: make minor ticks gray
    p.yaxis.minor_tick_line_color = "gray"

    # Clean look
    p.xaxis.major_label_text_font_size = "0pt"
    p.yaxis.major_label_text_font_size = "0pt"
    p.xaxis.major_label_orientation = 0.8
    p.xgrid.visible = False
    p.ygrid.visible = False

    # ---- Colormap ----
    cmap = plt.colormaps["Blues"]
    blues = [to_hex(cmap(i)) for i in np.linspace(0, 1, 256)]
    p.rect(
        x="i",
        y="j",
        width=1,
        height=1,
        source=df_AB,
        fill_color=linear_cmap(
            'value', blues,
            low=df_AB.value.min(),
            high=np.percentile(df_AB.value, 99.8)
        ),
        line_color=None
    )

    p.xaxis.axis_label = "Protein A"
    p.yaxis.axis_label = "Protein B"

    return p

In [ ]:
from bokeh.plotting import figure, show
from bokeh.transform import linear_cmap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex


def plot_full_contact(contact, tokens, len_A, major_step=10):

    start_A = 3
    end_A = 3 + len_A

    start_B = end_A + 3
    end_B = len(tokens)

    n = contact.shape[0]

    # ---- Dataframe with numeric coordinates ----
    rows, cols = np.indices(contact.shape)

    df = pd.DataFrame({
        "i": cols.ravel() + 1,
        "j": rows.ravel() + 1,
        "value": contact.ravel()
    })

    # ---- Figure ----
    p = figure(
        title="Full-Length Contact Map",
        width=900,
        height=900,
        x_range=(0.5, n + 0.5),
        y_range=(n + 0.5, 0.5),
        tools="hover,save,pan,box_zoom,reset,wheel_zoom",
        tooltips=[
            ("Position 1", "@i"),
            ("Position 2", "@j"),
            ("value", "@value")
        ]
    )

    # ---- Major ticks ----
    ticks = np.arange(1, n + 1, major_step)

    p.xaxis.ticker = ticks
    p.yaxis.ticker = ticks

    # ---- Remove grid ----
    p.xgrid.visible = False
    p.ygrid.visible = False

    # ---- Colormap ----
    cmap = plt.colormaps["Blues"]
    blues = [to_hex(cmap(i)) for i in np.linspace(0, 1, 256)]

    p.rect(
        x="i",
        y="j",
        width=1,
        height=1,
        source=df,
        fill_color=linear_cmap(
            "value",
            blues,
            low=df.value.min(),
            high=np.percentile(df.value, 99.8)
        ),
        line_color=None
    )

    # ==========================================================
    # A/B SEPARATION
    # ==========================================================

    # Boundary after A
    boundary = end_A + 0.5

    # Vertical
    p.line(
        x=[boundary, boundary],
        y=[0.5, n + 0.5],
        line_color="red",
        line_width=2
    )

    # Horizontal
    p.line(
        x=[0.5, n + 0.5],
        y=[boundary, boundary],
        line_color="red",
        line_width=2
    )

    p.xaxis.axis_label = "Sequence position"
    p.yaxis.axis_label = "Sequence position"

    return p

In [ ]:
p_AB = plot_A_vs_B(contact, tokens, len(A), major_step=10)
show(p_AB)

In [ ]:
p_full = plot_full_contact(
    contact,
    tokens,
    len(A),
    major_step=10
)

show(p_full)